In [1]:
import os
import torch
import numpy as np
import argparse
import torch
import torch.nn as nn
import wandb
import polars as pl

from utils.general_utils import set_seed
from dvrl.dataset import EssayDataset
from models.paes import PAES
from dvrl.predictor_config import PAESModelConfig
from dvrl.fn_predictor import fit_func, pred_func, calc_qwk

/home/ito/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [44]:
target_prompt_id = 8
device = torch.device('cuda')
set_seed(12)

In [45]:
###################################################
# Step1. Load Data
###################################################
# Load essay data
print('Loading essay data...')
dataset = EssayDataset('../data/training_set_rel3.xlsx', '../data/hand_crafted_v3.csv', '../data/readability_features.csv')
dataset.preprocess_dataframe()
train_data, dev_data, test_data = dataset.cross_prompt_split(
    target_prompt_set=target_prompt_id,
    dev_size=0,
    cache_dir='.embedding_cache',
    embedding_model='microsoft/deberta-v3-large',
    add_pos=False,
)
print(f'    Number of training samples: {len(train_data["essay_id"])}')
print(f'    Number of dev samples: {len(dev_data["essay_id"])}')
print(f'    Number of test samples: {len(test_data["essay_id"])}')

Loading essay data...


/home/ito/.local/lib/python3.10/site-packages/huggingface_hub/file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
0it [00:00, ?it/s]
0it [00:00, ?it/s]

    Number of training samples: 12254
    Number of dev samples: 0
    Number of test samples: 723


# Run PAES

In [ ]:
from sklearn.model_selection import train_test_split

train_index = np.array(range(len(train_data['essay_id'])))

# 擬似ラベルを付与するデータとしないデータに分割
train_index, val_index = train_test_split(
    train_index,
    test_size=0.2,
    random_state=12,
    shuffle=True
)

config = PAESModelConfig()
model = PAES(train_data['max_sentnum'], train_data['max_sentlen'], train_data['pos_vocab']).to(device)
# train data
x_train = [train_data['pos_x'][train_index], train_data['feature'][train_index], train_data['readability'][train_index]]
y_train = train_data['scaled_score'][train_index].reshape(-1, 1)
# dev data
x_dev = [train_data['pos_x'][val_index], train_data['feature'][val_index], train_data['readability'][val_index]]
y_dev = train_data['scaled_score'][val_index].reshape(-1, 1)
# test data
x_test = [
    np.concatenate([test_data['pos_x'], dev_data['pos_x']], axis=0),
    np.concatenate([test_data['feature'], dev_data['feature']], axis=0),
    np.concatenate([test_data['readability'], dev_data['readability']], axis=0)
]
y_test = np.concatenate([test_data['scaled_score'], dev_data['scaled_score']])

model = fit_func(
    model,
    x_train,
    y_train,
    config.optimizer,
    config.lr,
    config.batch_size,
    config.epochs,
    device,
    target_prompt_id,
    'mse',
    x_dev,
    y_dev,
    False,
    verbose=True
)

# Predict
print('Predicting...')
y_dev_pred = pred_func(
    model,
    x_dev,
    config.batch_size,
    device
)
y_test_pred = pred_func(
    model,
    x_test,
    config.batch_size,
    device
)

# Calculate QWK
print('Calculating QWK...')
# dev_qwk = calc_qwk(y_dev, y_dev_pred, target_prompt_id, 'score')
test_qwk = calc_qwk(y_test, y_test_pred, target_prompt_id, 'score')

# print(f'    Dev QWK: {dev_qwk}')
print(f'    Test QWK: {test_qwk}')

Epoch 1/50
dev mse: 0.02645481564104557
Epoch 2/50
dev mse: 0.027028998360037804
Epoch 3/50


KeyboardInterrupt: 

# Run Features model

In [54]:
from models.features import FeaturesModel
from utils.dvrl_utils import fit_func, pred_func, calc_qwk

model = FeaturesModel().to(device)

fit_func(
    model,
    np.concatenate([train_data['feature'], train_data['readability']], axis=1),
    train_data['scaled_score'],
    512,
    500,
    device,
)

y_test_pred = pred_func(
    model,
    np.concatenate([test_data['feature'], test_data['readability']], axis=1),
    512,
    device
)

# Calculate QWK
test_qwk = calc_qwk(test_data['scaled_score'], y_test_pred, target_prompt_id, 'score')

print(f'Test QWK: {test_qwk}')

Test QWK: 0.6307564200979902


In [55]:
import polars as pl
# y_test_predの値をCSVで保存
df = pl.DataFrame({
    'essay_id': test_data['essay_id'],
    'y_pred': y_test_pred.flatten()
})
df.write_csv(f'../outputs/features_model/prediction_{target_prompt_id}.csv')

In [22]:
from utils.general_utils import get_min_max_scores

In [56]:
# dfを読み込み
df_list = []
for i in range(1, 9):
    df = pl.read_csv(f'../outputs/features_model/prediction_{i}.csv')
    minscore, maxscore = get_min_max_scores()[i]['score']
    df = df.with_columns(
        (pl.col('y_pred') * (maxscore - minscore) + minscore).round().alias('y_pred_rescaled')
    )
    df_list.append(df)

# df_listを結合
df = pl.concat(df_list)

# dfをessay_idでソート
df = df.sort('essay_id')

# dfを保存
df.write_csv(f'../outputs/features_model/prediction_all.csv')